# Section 1: Project Setup and Dependencies


In [1]:
import os
import sys
import sqlite3
import json
import re
import urllib.request
import urllib.parse
import datetime
import time
import contextvars
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
print("All project dependencies loaded successfully.")


All project dependencies loaded successfully.


# Section 2: Environment Configuration and Groq API Key


In [2]:
load_dotenv()
load_dotenv(os.path.join("..", ".env"))

groq_api_key = os.environ.get("GROQ_API_KEY")

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    groq_api_key = user_secrets.get_secret("GROQ_API_KEY")
except Exception:
    pass

if not groq_api_key:
    print("GROQ_API_KEY is not configured.")
    print("In Kaggle, navigate to Add-ons -> Secrets -> Add a new secret with Label: GROQ_API_KEY.")
    print("Alternatively, define GROQ_API_KEY in your local environment or .env file.")
else:
    os.environ["GROQ_API_KEY"] = groq_api_key
    masked_key = groq_api_key[:4] + "*" * (len(groq_api_key) - 8) + groq_api_key[-4:]
    print("Groq API Key detected and configured successfully: " + masked_key)


Groq API Key detected and configured successfully: gsk_************************************************Two9


# Section 3: SQLite Database Creation and Seeding


In [3]:
database_file = "lab_test_database.db"

def initialize_database(db_path=database_file):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("DROP TABLE IF EXISTS bookings")
    cursor.execute("DROP TABLE IF EXISTS slots")
    cursor.execute("DROP TABLE IF EXISTS tests")

    cursor.execute('''
    CREATE TABLE tests (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT UNIQUE NOT NULL,
        category TEXT NOT NULL,
        price REAL NOT NULL,
        description TEXT
    )
    ''')

    cursor.execute('''
    CREATE TABLE slots (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        test_id INTEGER NOT NULL REFERENCES tests(id),
        slot_date TEXT NOT NULL,
        slot_time TEXT NOT NULL,
        is_available INTEGER NOT NULL DEFAULT 1,
        UNIQUE(test_id, slot_date, slot_time)
    )
    ''')

    cursor.execute('''
    CREATE TABLE bookings (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        booking_reference TEXT UNIQUE NOT NULL,
        user_id TEXT NOT NULL,
        test_id INTEGER NOT NULL REFERENCES tests(id),
        slot_id INTEGER NOT NULL REFERENCES slots(id),
        status TEXT NOT NULL DEFAULT 'CONFIRMED',
        created_at TEXT NOT NULL
    )
    ''')

    diagnostic_tests = [
        ("Complete Blood Count (CBC)", "Blood Tests", 450.0, "Evaluates overall health and detects anemia and infections."),
        ("Lipid Profile", "Blood Tests", 850.0, "Measures cholesterol and triglyceride levels for cardiovascular assessment."),
        ("HbA1c Glycated Hemoglobin", "Blood Tests", 600.0, "Measures average blood sugar levels over past 3 months."),
        ("Thyroid Stimulating Hormone (TSH)", "Blood Tests", 500.0, "Assesses thyroid gland function."),
        ("Chest X-Ray", "Imaging & Radiology", 750.0, "Produces radiographic images of chest organs."),
        ("Abdominal Ultrasound", "Imaging & Radiology", 1500.0, "Ultrasonic imaging of abdominal organs."),
        ("Brain MRI", "Imaging & Radiology", 6500.0, "Detailed MRI scans of brain structures."),
        ("Electrocardiogram (ECG)", "Cardiology & Pathology", 400.0, "Records electrical heart activity."),
        ("2D Echocardiogram", "Cardiology & Pathology", 2200.0, "Ultrasound examination of heart valves and chambers.")
    ]

    for name, category, price, desc in diagnostic_tests:
        cursor.execute("INSERT INTO tests (name, category, price, description) VALUES (?, ?, ?, ?)", (name, category, price, desc))
        test_id = cursor.lastrowid

        if name == "2D Echocardiogram":
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-20", "10:00 AM", 0))
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-21", "01:00 PM", 0))
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-22", "04:00 PM", 0))
        else:
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-20", "09:00 AM", 1))
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-21", "11:30 AM", 1))
            cursor.execute("INSERT INTO slots (test_id, slot_date, slot_time, is_available) VALUES (?, ?, ?, ?)", (test_id, "2026-11-22", "03:00 PM", 1))

    conn.commit()
    conn.close()
    print("Database initialized and seeded successfully.")

initialize_database()


Database initialized and seeded successfully.


# Section 4: Database Verification


In [4]:
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

cursor.execute("SELECT COUNT(*), COUNT(DISTINCT category) FROM tests")
test_count, category_count = cursor.fetchone()
print(f"Total Diagnostic Tests: {test_count}")
print(f"Total Categories: {category_count}")

cursor.execute("SELECT category, COUNT(*) FROM tests GROUP BY category")
for cat, count in cursor.fetchall():
    print(f"  - {cat}: {count} tests")

cursor.execute("SELECT t.name, COUNT(s.id), SUM(s.is_available) FROM tests t JOIN slots s ON t.id = s.test_id GROUP BY t.name")
print("\nSlots Availability Breakdown:")
for tname, total_slots, avail_slots in cursor.fetchall():
    print(f"  - {tname}: {total_slots} total slots, {avail_slots} available")

cursor.execute("SELECT COUNT(*) FROM bookings")
print(f"\nInitial Bookings Count: {cursor.fetchone()[0]}")
conn.close()


Total Diagnostic Tests: 9
Total Categories: 3
  - Blood Tests: 4 tests
  - Cardiology & Pathology: 2 tests
  - Imaging & Radiology: 3 tests

Slots Availability Breakdown:
  - 2D Echocardiogram: 3 total slots, 0 available
  - Abdominal Ultrasound: 3 total slots, 3 available
  - Brain MRI: 3 total slots, 3 available
  - Chest X-Ray: 3 total slots, 3 available
  - Complete Blood Count (CBC): 3 total slots, 3 available
  - Electrocardiogram (ECG): 3 total slots, 3 available
  - HbA1c Glycated Hemoglobin: 3 total slots, 3 available
  - Lipid Profile: 3 total slots, 3 available
  - Thyroid Stimulating Hormone (TSH): 3 total slots, 3 available

Initial Bookings Count: 0


# Section 5: Read Tools


In [5]:
current_user_context = contextvars.ContextVar("current_user_context", default="user_default")

def get_current_user_id():
    return current_user_context.get()

def set_current_user_id(user_id):
    current_user_context.set(user_id)

@tool(description="List available diagnostic tests filtered optionally by category name")
def list_tests(category: str = "") -> str:
    conn = sqlite3.connect(database_file)
    cursor = conn.cursor()
    if category and str(category).strip() and str(category).lower() not in ["none", "all"]:
        cursor.execute(
            "SELECT id, name, category, price, description FROM tests WHERE LOWER(category) LIKE ? ORDER BY id",
            ("%" + str(category).strip().lower() + "%",)
        )
    else:
        cursor.execute("SELECT id, name, category, price, description FROM tests ORDER BY id")
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return "No tests found matching the requested category."

    results = []
    for test_id, name, cat, price, desc in rows:
        results.append(f"Test ID {test_id}: {name} [{cat}] - Price: Rs {price:.2f}\n  Description: {desc}")
    return "\n\n".join(results)

@tool(description="Check available time slots for a specific diagnostic test using its name or ID")
def check_slots(test_name_or_id: str) -> str:
    conn = sqlite3.connect(database_file)
    cursor = conn.cursor()

    cleaned_query = str(test_name_or_id).strip()
    if cleaned_query.isdigit():
        cursor.execute("SELECT id, name FROM tests WHERE id = ?", (int(cleaned_query),))
    else:
        cursor.execute("SELECT id, name FROM tests WHERE LOWER(name) LIKE ?", ("%" + cleaned_query.lower() + "%",))

    test_row = cursor.fetchone()
    if not test_row:
        conn.close()
        return f"Diagnostic test '{test_name_or_id}' was not found in the lab catalog."

    test_id, test_name = test_row
    cursor.execute(
        "SELECT id, slot_date, slot_time FROM slots WHERE test_id = ? AND is_available = 1 ORDER BY slot_date, slot_time",
        (test_id,)
    )
    available_slots = cursor.fetchall()
    conn.close()

    if not available_slots:
        return f"No available slots found for {test_name}. All slots are currently fully booked."

    slot_entries = [f"Slot ID {sid}: Date {sdate} at {stime}" for sid, sdate, stime in available_slots]
    return f"Available slots for {test_name} (Test ID: {test_id}):\n" + "\n".join(slot_entries)

@tool(description="View all confirmed bookings belonging to the currently authenticated patient")
def my_bookings() -> str:
    user_id = get_current_user_id()
    conn = sqlite3.connect(database_file)
    cursor = conn.cursor()
    cursor.execute('''
        SELECT b.booking_reference, t.name, t.category, s.slot_date, s.slot_time, t.price, b.status, b.created_at
        FROM bookings b
        JOIN tests t ON b.test_id = t.id
        JOIN slots s ON b.slot_id = s.id
        WHERE b.user_id = ? AND b.status = 'CONFIRMED'
        ORDER BY b.id DESC
    ''', (user_id,))
    rows = cursor.fetchall()
    conn.close()

    if not rows:
        return f"No active confirmed bookings found for patient '{user_id}'."

    entries = []
    for ref, tname, cat, sdate, stime, price, status, created in rows:
        entries.append(
            f"Reference: {ref}\n  Test: {tname} ({cat})\n  Schedule: {sdate} at {stime}\n  Price: Rs {price:.2f}\n  Status: {status}\n  Booked On: {created}"
        )
    return f"Confirmed Bookings for {user_id}:\n\n" + "\n\n".join(entries)

print("Read tools defined successfully.")


Read tools defined successfully.


In [6]:
set_current_user_id("user_alice")
print("1. list_tests output (Blood Tests):")
print(list_tests.invoke({"category": "Blood Tests"}))
print("\n" + "=" * 50 + "\n")
print("2. check_slots for Lipid Profile:")
print(check_slots.invoke({"test_name_or_id": "Lipid Profile"}))
print("\n" + "=" * 50 + "\n")
print("3. check_slots for fully booked 2D Echocardiogram:")
print(check_slots.invoke({"test_name_or_id": "2D Echocardiogram"}))
print("\n" + "=" * 50 + "\n")
print("4. my_bookings for Alice:")
print(my_bookings.invoke({}))


1. list_tests output (Blood Tests):
Test ID 1: Complete Blood Count (CBC) [Blood Tests] - Price: Rs 450.00
  Description: Evaluates overall health and detects anemia and infections.

Test ID 2: Lipid Profile [Blood Tests] - Price: Rs 850.00
  Description: Measures cholesterol and triglyceride levels for cardiovascular assessment.

Test ID 3: HbA1c Glycated Hemoglobin [Blood Tests] - Price: Rs 600.00
  Description: Measures average blood sugar levels over past 3 months.

Test ID 4: Thyroid Stimulating Hormone (TSH) [Blood Tests] - Price: Rs 500.00
  Description: Assesses thyroid gland function.


2. check_slots for Lipid Profile:
Available slots for Lipid Profile (Test ID: 2):
Slot ID 4: Date 2026-11-20 at 09:00 AM
Slot ID 5: Date 2026-11-21 at 11:30 AM
Slot ID 6: Date 2026-11-22 at 03:00 PM


3. check_slots for fully booked 2D Echocardiogram:
No available slots found for 2D Echocardiogram. All slots are currently fully booked.


4. my_bookings for Alice:
No active confirmed bookings fo

# Section 6: Write Tools


In [7]:
@tool(description="Book an available slot for a diagnostic test by specifying test name or ID and slot ID")
def book_test(test_name_or_id: str, slot_id: int) -> str:
    user_id = get_current_user_id()
    conn = sqlite3.connect(database_file)
    cursor = conn.cursor()

    cleaned_query = str(test_name_or_id).strip()
    if cleaned_query.isdigit():
        cursor.execute("SELECT id, name, price, category FROM tests WHERE id = ?", (int(cleaned_query),))
    else:
        cursor.execute("SELECT id, name, price, category FROM tests WHERE LOWER(name) LIKE ?", ("%" + cleaned_query.lower() + "%",))

    test_row = cursor.fetchone()
    if not test_row:
        conn.close()
        return f"Booking failed: Diagnostic test '{test_name_or_id}' was not found."

    test_id, test_name, price, category = test_row

    cursor.execute("SELECT id, test_id, slot_date, slot_time, is_available FROM slots WHERE id = ?", (int(slot_id),))
    slot_row = cursor.fetchone()
    if not slot_row:
        conn.close()
        return f"Booking failed: Slot ID {slot_id} does not exist."

    sid, stest_id, slot_date, slot_time, is_avail = slot_row
    if stest_id != test_id:
        conn.close()
        return f"Booking failed: Slot ID {slot_id} does not belong to test '{test_name}'."

    if is_avail == 0:
        conn.close()
        return f"Booking failed: Slot ID {slot_id} is already booked or unavailable."

    cursor.execute('''
        SELECT id FROM bookings
        WHERE user_id = ? AND test_id = ? AND status = 'CONFIRMED'
    ''', (user_id, test_id))
    if cursor.fetchone():
        conn.close()
        return f"Booking failed: Patient '{user_id}' already has an active confirmed booking for '{test_name}'."

    booking_reference = f"BKG-{int(datetime.datetime.now().timestamp()) % 1000000:06d}"

    try:
        cursor.execute("UPDATE slots SET is_available = 0 WHERE id = ? AND is_available = 1", (int(slot_id),))
        if cursor.rowcount == 0:
            conn.rollback()
            conn.close()
            return f"Booking conflict: Slot ID {slot_id} was just booked by another request."

        created_timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        cursor.execute('''
            INSERT INTO bookings (booking_reference, user_id, test_id, slot_id, status, created_at)
            VALUES (?, ?, ?, ?, 'CONFIRMED', ?)
        ''', (booking_reference, user_id, test_id, int(slot_id), created_timestamp))
        new_booking_id = cursor.lastrowid
        conn.commit()

        cursor.execute('''
            SELECT b.booking_reference, b.user_id, t.name, t.category, s.slot_date, s.slot_time, t.price, b.status, b.created_at
            FROM bookings b
            JOIN tests t ON b.test_id = t.id
            JOIN slots s ON b.slot_id = s.id
            WHERE b.id = ?
        ''', (new_booking_id,))
        receipt_row = cursor.fetchone()
        conn.close()

        ref, uid, tname, cat, sdate, stime, tprice, bstatus, bcreated = receipt_row
        receipt = (
            "========================================\n"
            "       LAB TEST BOOKING RECEIPT         \n"
            "========================================\n"
            f"Booking Reference: {ref}\n"
            f"Patient ID:        {uid}\n"
            f"Diagnostic Test:   {tname} ({cat})\n"
            f"Scheduled Date:    {sdate}\n"
            f"Scheduled Time:    {stime}\n"
            f"Total Price:       Rs {tprice:.2f}\n"
            f"Booking Status:    {bstatus}\n"
            f"Generated At:      {bcreated}\n"
            "========================================"
        )
        return receipt
    except Exception as e:
        conn.rollback()
        conn.close()
        return f"Booking failed due to database error: {str(e)}"

@tool(description="Cancel an active confirmed booking using its booking reference")
def cancel_booking(booking_reference: str) -> str:
    user_id = get_current_user_id()
    conn = sqlite3.connect(database_file)
    cursor = conn.cursor()

    cleaned_ref = booking_reference.strip().upper()
    cursor.execute('''
        SELECT b.id, b.user_id, b.slot_id, b.status, t.name
        FROM bookings b
        JOIN tests t ON b.test_id = t.id
        WHERE UPPER(b.booking_reference) = ?
    ''', (cleaned_ref,))
    row = cursor.fetchone()

    if not row:
        conn.close()
        return f"Cancellation failed: Booking reference '{booking_reference}' was not found."

    bid, owner_id, slot_id, status, test_name = row

    if owner_id != user_id:
        conn.close()
        return f"Cancellation rejected: Booking '{booking_reference}' belongs to another patient. You can only cancel your own bookings."

    if status != "CONFIRMED":
        conn.close()
        return f"Cancellation rejected: Booking '{booking_reference}' is already {status}."

    try:
        cursor.execute("UPDATE bookings SET status = 'CANCELLED' WHERE id = ?", (bid,))
        cursor.execute("UPDATE slots SET is_available = 1 WHERE id = ?", (slot_id,))
        conn.commit()

        cursor.execute('''
            SELECT b.booking_reference, b.status, t.name, s.slot_date, s.slot_time
            FROM bookings b
            JOIN tests t ON b.test_id = t.id
            JOIN slots s ON b.slot_id = s.id
            WHERE b.id = ?
        ''', (bid,))
        updated_row = cursor.fetchone()
        conn.close()

        ref, updated_status, tname, sdate, stime = updated_row
        receipt = (
            "========================================\n"
            "     BOOKING CANCELLATION NOTICE        \n"
            "========================================\n"
            f"Booking Reference: {ref}\n"
            f"Patient ID:        {user_id}\n"
            f"Diagnostic Test:   {tname}\n"
            f"Released Slot:     {sdate} at {stime}\n"
            f"Updated Status:    {updated_status}\n"
            "========================================"
        )
        return receipt
    except Exception as e:
        conn.rollback()
        conn.close()
        return f"Cancellation failed due to database error: {str(e)}"

print("Write tools defined successfully.")


Write tools defined successfully.


# Section 7: User Identity and Tool Schema


In [8]:
tools_list = [list_tests, check_slots, my_bookings, book_test, cancel_booking]

print("Model-Visible Tool Schema Inspection:")
print("=" * 60)

for t in tools_list:
    print(f"Tool Name: {t.name}")
    print(f"Description: {t.description}")
    print(f"Model-Controlled Arguments: {list(t.args.keys())}")
    has_user_id = "user_id" in t.args
    print(f"Is 'user_id' in Model Arguments? {has_user_id} (Expected: False)")
    print("-" * 60)

print("\nRuntime Context Demonstration:")
set_current_user_id("user_alice")
print(f"Active runtime user context: {get_current_user_id()}")
set_current_user_id("user_bob")
print(f"Switched runtime user context: {get_current_user_id()}")


Model-Visible Tool Schema Inspection:
Tool Name: list_tests
Description: List available diagnostic tests filtered optionally by category name
Model-Controlled Arguments: ['category']
Is 'user_id' in Model Arguments? False (Expected: False)
------------------------------------------------------------
Tool Name: check_slots
Description: Check available time slots for a specific diagnostic test using its name or ID
Model-Controlled Arguments: ['test_name_or_id']
Is 'user_id' in Model Arguments? False (Expected: False)
------------------------------------------------------------
Tool Name: my_bookings
Description: View all confirmed bookings belonging to the currently authenticated patient
Model-Controlled Arguments: []
Is 'user_id' in Model Arguments? False (Expected: False)
------------------------------------------------------------
Tool Name: book_test
Description: Book an available slot for a diagnostic test by specifying test name or ID and slot ID
Model-Controlled Arguments: ['test_

# Section 8: Web Search Integration


In [9]:
medical_preparation_knowledge = {
    "lipid profile": "A Lipid Profile blood test typically requires a 9 to 12 hour overnight fast before the sample is drawn. You may drink plain water, but should avoid food, alcohol, and caffeine.",
    "complete blood count": "A standard Complete Blood Count (CBC) usually does not require fasting unless your doctor has paired it with a glucose or lipid panel.",
    "cbc": "A standard Complete Blood Count (CBC) generally does not require fasting. Normal hydration with water is recommended.",
    "hba1c": "An HbA1c test measures the average level of blood glucose over the previous 2 to 3 months. Fasting is not required for an HbA1c test.",
    "tsh": "A Thyroid Stimulating Hormone (TSH) test usually does not require fasting, but morning testing is often preferred as TSH levels fluctuate throughout the day.",
    "chest x-ray": "No fasting is required for a Chest X-Ray. Patients must remove metal objects, jewelry, and piercings around the chest area.",
    "abdominal ultrasound": "An abdominal ultrasound typically requires fasting for 8 to 12 hours before the examination to prevent gas buildup in the digestive tract.",
    "brain mri": "Brain MRI examinations do not typically require fasting unless contrast dye is being administered. All magnetic and metallic items must be removed.",
    "ecg": "An Electrocardiogram (ECG) does not require fasting. Avoid cold water or heavy physical exertion immediately prior to the test.",
    "echocardiogram": "A standard transthoracic echocardiogram requires no fasting. You can eat, drink, and take your usual medications prior to the test."
}

@tool(description="Search for general medical test preparation, fasting requirements, or what a diagnostic test measures. Never use this for prices or slot availability.")
def web_search_health_info(query: str) -> str:
    cleaned_query = query.strip().lower()

    for key, info in medical_preparation_knowledge.items():
        if key in cleaned_query:
            return f"General Health Information: {info}"

    try:
        encoded_query = urllib.parse.quote_plus(query)
        api_url = f"https://api.duckduckgo.com/?q={encoded_query}&format=json&no_html=1&skip_disambig=1"
        request = urllib.request.Request(api_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=3) as response:
            data = json.loads(response.read().decode("utf-8"))
            abstract = data.get("AbstractText") or data.get("Abstract")
            if abstract and len(abstract.strip()) > 20:
                return f"Web Search Result: {abstract.strip()}"
    except Exception:
        pass

    return "General Health Information: For specific diagnostic preparation guidelines, please follow the preparation instructions provided by your referring physician."

print("Web search tool defined successfully.")
print("\nTest Web Search (Fasting for Lipid Profile):")
print(web_search_health_info.invoke({"query": "Do I need fasting for Lipid Profile?"}))


Web search tool defined successfully.

Test Web Search (Fasting for Lipid Profile):
General Health Information: A Lipid Profile blood test typically requires a 9 to 12 hour overnight fast before the sample is drawn. You may drink plain water, but should avoid food, alcohol, and caffeine.


# Section 9: Agent Creation and Memory


In [10]:
system_instruction = (
    "You are an AI Lab Test Booking Assistant for a diagnostic healthcare clinic.\n"
    "Your responsibilities and strict constraints are:\n"
    "1. Live Database Only: Always obtain diagnostic test names, categories, prices, and available slots from the database tools (list_tests, check_slots, my_bookings). Never fabricate or guess prices, availability, or booking references.\n"
    "2. Tool Invocations: When a patient requests or confirms booking or cancelling a test (e.g. 'book slot X', 'please book', 'I confirm', 'cancel booking X'), invoke the book_test or cancel_booking tool immediately. The execution framework enforces human-in-the-loop authorization.\n"
    "3. No Diagnosis: NEVER diagnose symptoms, prescribe treatments, or suggest specific diagnostic tests for self-reported symptoms. If a user asks about symptoms or medical issues, politely inform them that you cannot provide medical diagnoses and advise them to consult a qualified physician immediately.\n"
    "4. General Health Info: Use the web_search_health_info tool solely for general educational inquiries like fasting instructions or test preparation.\n"
    "5. Be concise, respectful, professional, and clear."
)

all_assistant_tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    web_search_health_info
]

assistant_tool_map = {t.name: t for t in all_assistant_tools}

def create_agent(model_name="openai/gpt-oss-120b", temperature=0.0):
    llm = ChatGroq(model=model_name, temperature=temperature)
    model_with_tools = llm.bind_tools(all_assistant_tools)
    return {
        "model": model_with_tools,
        "tools": assistant_tool_map,
        "system_prompt": system_instruction
    }

user_conversation_memories = {}

def get_user_memory(user_id):
    if user_id not in user_conversation_memories:
        user_conversation_memories[user_id] = [SystemMessage(content=system_instruction)]
    return user_conversation_memories[user_id]

def clear_user_memory(user_id):
    user_conversation_memories[user_id] = [SystemMessage(content=system_instruction)]

booking_agent = create_agent()
print("Lab Test Booking Agent and per-user memory initialized successfully.")


Lab Test Booking Agent and per-user memory initialized successfully.


# Section 10: Human Approval Mechanism


In [11]:
def execute_agent_turn(agent, user_id, user_message, approval_decision=None, interactive=False):
    set_current_user_id(user_id)
    history = get_user_memory(user_id)

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    redacted_message = re.sub(email_pattern, '[REDACTED_EMAIL]', user_message)

    history.append(HumanMessage(content=redacted_message))

    max_steps = 5
    step = 0
    executed_tools = []
    approval_records = []

    while step < max_steps:
        step += 1
        response = agent["model"].invoke(history)
        history.append(response)

        if not response.tool_calls:
            return {
                "response": response.content,
                "executed_tools": executed_tools,
                "approval_records": approval_records,
                "redacted_message": redacted_message
            }

        for call in response.tool_calls:
            tool_name = call["name"]
            tool_args = call["args"]
            tool_id = call["id"]
            executed_tools.append(tool_name)

            if tool_name in ["book_test", "cancel_booking"]:
                if interactive:
                    decision_input = input(f"[HUMAN APPROVAL REQUIRED] Authorize {tool_name} with args {tool_args}? (approve/reject): ").strip().lower()
                    approved = decision_input in ["approve", "yes", "y"]
                elif approval_decision is not None:
                    approved = bool(approval_decision)
                else:
                    approved = True

                approval_records.append({"tool": tool_name, "args": tool_args, "approved": approved})

                if approved:
                    tool_fn = agent["tools"][tool_name]
                    tool_result = tool_fn.invoke(tool_args)
                else:
                    tool_result = (
                        f"Action '{tool_name}' was REJECTED by the user during human approval. "
                        "The operation was aborted and no database changes were made."
                    )
            else:
                tool_fn = agent["tools"][tool_name]
                tool_result = tool_fn.invoke(tool_args)

            history.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))

    return {
        "response": history[-1].content,
        "executed_tools": executed_tools,
        "approval_records": approval_records,
        "redacted_message": redacted_message
    }

print("Approval execution mechanism configured.")


Approval execution mechanism configured.


In [12]:
initialize_database()
set_current_user_id("user_test_approval")
clear_user_memory("user_test_approval")

conn = sqlite3.connect(database_file)
c = conn.cursor()
c.execute("SELECT COUNT(*) FROM bookings")
initial_bookings = c.fetchone()[0]

print("=== Demonstration 1: Human Rejection ===")
rejection_turn = execute_agent_turn(
    booking_agent,
    "user_test_approval",
    "I confirm and authorize: please book slot 1 for Complete Blood Count (CBC) now.",
    approval_decision=False
)
print("Tool Calls:", rejection_turn["executed_tools"])
print("Approval Records:", rejection_turn["approval_records"])
print("Assistant Response:\n", rejection_turn["response"])

c.execute("SELECT COUNT(*) FROM bookings")
after_rejection_bookings = c.fetchone()[0]
c.execute("SELECT is_available FROM slots WHERE id = 1")
slot_1_avail = c.fetchone()[0]
print(f"Database Verification after Rejection:")
print(f"  Bookings Count: {initial_bookings} -> {after_rejection_bookings} (No change: {initial_bookings == after_rejection_bookings})")
print(f"  Slot 1 Availability: {slot_1_avail} (Remains available)")

print("\n" + "=" * 60 + "\n")

print("=== Demonstration 2: Human Approval ===")
clear_user_memory("user_test_approval")
approval_turn = execute_agent_turn(
    booking_agent,
    "user_test_approval",
    "I confirm and authorize: please book slot 1 for Complete Blood Count (CBC) now.",
    approval_decision=True
)
print("Tool Calls:", approval_turn["executed_tools"])
print("Approval Records:", approval_turn["approval_records"])
print("Assistant Response:\n", approval_turn["response"])

c.execute("SELECT COUNT(*) FROM bookings")
after_approval_bookings = c.fetchone()[0]
c.execute("SELECT is_available FROM slots WHERE id = 1")
slot_1_avail_after = c.fetchone()[0]
print(f"Database Verification after Approval:")
print(f"  Bookings Count: {initial_bookings} -> {after_approval_bookings} (Added 1 booking: {after_approval_bookings == initial_bookings + 1})")
print(f"  Slot 1 Availability: {slot_1_avail_after} (Slot marked unavailable)")
conn.close()


Database initialized and seeded successfully.
=== Demonstration 1: Human Rejection ===
Tool Calls: ['check_slots', 'book_test']
Approval Records: [{'tool': 'book_test', 'args': {'slot_id': 1, 'test_name_or_id': 'Complete Blood Count (CBC)'}, 'approved': False}]
Assistant Response:
 I’m sorry, but the booking could not be completed because the request was not approved. If you’d like to try a different time slot or would like me to attempt the booking again, please let me know which slot you’d prefer.
Database Verification after Rejection:
  Bookings Count: 0 -> 0 (No change: True)
  Slot 1 Availability: 1 (Remains available)


=== Demonstration 2: Human Approval ===
Tool Calls: ['book_test']
Approval Records: [{'tool': 'book_test', 'args': {'slot_id': 1, 'test_name_or_id': 'Complete Blood Count (CBC)'}, 'approved': True}]
Assistant Response:
 Your booking has been confirmed!

**Booking Reference:** BKG-923287  
**Test:** Complete Blood Count (CBC) – Blood Tests  
**Scheduled Date & Time

# Section 11: Reliability and Retry Demonstration


In [13]:
simulate_transient_failure = True

def check_slots_with_retry(test_name_or_id: str, max_retries: int = 2) -> str:
    global simulate_transient_failure
    for attempt in range(1, max_retries + 1):
        try:
            if simulate_transient_failure:
                simulate_transient_failure = False
                raise sqlite3.OperationalError("database is locked (transient connection error)")
            return check_slots.invoke({"test_name_or_id": test_name_or_id})
        except sqlite3.OperationalError as error:
            print(f"[RETRY ATTEMPT {attempt}] Caught transient error: {error}. Initiating automatic retry...")
            time.sleep(0.1)

print("=== Controlled Reliability & Retry Demonstration ===")
simulate_transient_failure = True
result = check_slots_with_retry("Lipid Profile")
print("\nResult after automatic recovery:")
print(result)

print("\nVerifying normal operation after demonstration:")
normal_result = check_slots_with_retry("Complete Blood Count (CBC)")
print(normal_result)


=== Controlled Reliability & Retry Demonstration ===
[RETRY ATTEMPT 1] Caught transient error: database is locked (transient connection error). Initiating automatic retry...

Result after automatic recovery:
Available slots for Lipid Profile (Test ID: 2):
Slot ID 4: Date 2026-11-20 at 09:00 AM
Slot ID 5: Date 2026-11-21 at 11:30 AM
Slot ID 6: Date 2026-11-22 at 03:00 PM

Verifying normal operation after demonstration:
Available slots for Complete Blood Count (CBC) (Test ID: 1):
Slot ID 2: Date 2026-11-21 at 11:30 AM
Slot ID 3: Date 2026-11-22 at 03:00 PM


# Section 12: Email Redaction Demonstration


In [14]:
def redact_email_address(text: str) -> str:
    pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    return re.sub(pattern, '[REDACTED_EMAIL]', text)

raw_patient_message = "Hello, my email is patient.shubranil@example.com. Can you check what blood tests are available?"
redacted_patient_message = redact_email_address(raw_patient_message)

print("=== Email Redaction Verification ===")
print("Original User Input: ", raw_patient_message)
print("Redacted Model Input:", redacted_patient_message)
print("Is original email exposed in model input? ", "patient.shubranil@example.com" in redacted_patient_message)
print("Is [REDACTED_EMAIL] token present?       ", "[REDACTED_EMAIL]" in redacted_patient_message)

set_current_user_id("user_privacy_test")
clear_user_memory("user_privacy_test")
privacy_turn = execute_agent_turn(
    booking_agent,
    "user_privacy_test",
    raw_patient_message
)
print("\\nAgent Executed Turn with Redacted Input:")
print("Assistant Response:\\n", privacy_turn["response"])


=== Email Redaction Verification ===
Original User Input:  Hello, my email is patient.shubranil@example.com. Can you check what blood tests are available?
Redacted Model Input: Hello, my email is [REDACTED_EMAIL] Can you check what blood tests are available?
Is original email exposed in model input?  False
Is [REDACTED_EMAIL] token present?        True
\nAgent Executed Turn with Redacted Input:
Assistant Response:\n Here are the blood‑test options we currently offer:

| Test ID | Test Name | Price (Rs) | Brief Description |
|--------|-----------|------------|-------------------|
| 1 | Complete Blood Count (CBC) | 450.00 | Evaluates overall health and detects anemia or infections. |
| 2 | Lipid Profile | 850.00 | Measures cholesterol and triglyceride levels for cardiovascular assessment. |
| 3 | HbA1c (Glycated Hemoglobin) | 600.00 | Shows average blood‑sugar levels over the past three months. |
| 4 | Thyroid Stimulating Hormone (TSH) | 500.00 | Assesses thyroid gland function. |

Let m

# Section 13: Interactive Booking Assistant


In [15]:
def run_interactive_assistant(user_id="user_guest", preloaded_messages=None):
    set_current_user_id(user_id)
    clear_user_memory(user_id)
    print(f"Starting Interactive Lab Booking Assistant for Patient: {user_id}")
    print("Type 'exit', 'quit', or 'q' to end the session.\n" + "=" * 60)

    message_queue = list(preloaded_messages) if preloaded_messages else None

    while True:
        if message_queue is not None:
            if not message_queue:
                print("\n[Interactive Session: All queued inputs processed]")
                break
            user_input = message_queue.pop(0)
            print(f"\nPatient ({user_id}): {user_input}")
        else:
            user_input = input(f"\nPatient ({user_id}): ").strip()

        if user_input.lower() in ["exit", "quit", "q"]:
            print("Thank you for using the Lab Test Booking Assistant. Stay healthy!")
            break

        turn_output = execute_agent_turn(
            booking_agent,
            user_id,
            user_input,
            approval_decision=True,
            interactive=False
        )

        if turn_output["executed_tools"]:
            print(f"[Tool Calls]: {turn_output['executed_tools']}")

        if turn_output["approval_records"]:
            for rec in turn_output["approval_records"]:
                print(f"[Approval]: {rec['tool']} -> {'APPROVED' if rec['approved'] else 'REJECTED'}")

        print(f"Assistant:\n{turn_output['response']}")

demo_inputs = [
    "What blood tests do you offer?",
    "What slots are available for Complete Blood Count?",
    "exit"
]

run_interactive_assistant("user_demo", preloaded_messages=demo_inputs)


Starting Interactive Lab Booking Assistant for Patient: user_demo
Type 'exit', 'quit', or 'q' to end the session.

Patient (user_demo): What blood tests do you offer?
[Tool Calls]: ['list_tests']
Assistant:
Here are the blood‑test options we currently offer:

| Test ID | Test Name | Price (Rs) | Brief Description |
|--------|-----------|------------|-------------------|
| 1 | Complete Blood Count (CBC) | 450.00 | Evaluates overall health, detects anemia and infections. |
| 2 | Lipid Profile | 850.00 | Measures cholesterol and triglycerides for cardiovascular assessment. |
| 3 | HbA1c (Glycated Hemoglobin) | 600.00 | Shows average blood‑sugar levels over the past three months. |
| 4 | Thyroid Stimulating Hormone (TSH) | 500.00 | Assesses thyroid gland function. |

Let me know if you’d like more details about any test, want to see available appointment slots, or would like to book a test.

Patient (user_demo): What slots are available for Complete Blood Count?
[Tool Calls]: ['check_slots

# Section 14: Multi-User Privacy Demonstration


In [16]:
initialize_database()
clear_user_memory("user_alice")
clear_user_memory("user_bob")

print("=== Multi-User Privacy & Isolation Demonstration ===")

print("\n1. User Alice books slot 1 for Complete Blood Count (CBC):")
alice_turn = execute_agent_turn(
    booking_agent,
    "user_alice",
    "I confirm and authorize: please book slot 1 for Complete Blood Count (CBC) now.",
    approval_decision=True
)
print("Alice's booking result:\n", alice_turn["response"])

conn = sqlite3.connect(database_file)
c = conn.cursor()
c.execute("SELECT booking_reference FROM bookings WHERE user_id = 'user_alice'")
alice_booking_row = c.fetchone()
alice_booking_ref = alice_booking_row[0] if alice_booking_row else "BKG-UNKNOWN"
conn.close()
print(f"\nAlice's generated booking reference: {alice_booking_ref}")

print("\n2. User Bob checks his bookings:")
bob_bookings_turn = execute_agent_turn(
    booking_agent,
    "user_bob",
    "Show me my bookings"
)
print("Bob's bookings output:\n", bob_bookings_turn["response"])
print("Can Bob see Alice's booking? ", alice_booking_ref in bob_bookings_turn["response"])

print("\n3. User Bob attempts to cancel Alice's booking:")
bob_cancel_turn = execute_agent_turn(
    booking_agent,
    "user_bob",
    f"I confirm and authorize: please cancel booking {alice_booking_ref} now.",
    approval_decision=True
)
print("Bob's cancellation attempt:\n", bob_cancel_turn["response"])

print("\n4. User Alice verifies her booking remains active:")
alice_verify_turn = execute_agent_turn(
    booking_agent,
    "user_alice",
    "Show my bookings"
)
print("Alice's bookings verification:\n", alice_verify_turn["response"])
print("Is Alice's booking still confirmed? ", alice_booking_ref in alice_verify_turn["response"])


Database initialized and seeded successfully.
=== Multi-User Privacy & Isolation Demonstration ===

1. User Alice books slot 1 for Complete Blood Count (CBC):
Alice's booking result:
 Your booking is confirmed!

**Booking Reference:** BKG-923298  
**Test:** Complete Blood Count (CBC) – Blood Tests  
**Date & Time:** 20 Nov 2026 at 09:00 AM  

If you need to view, modify, or cancel this booking, just let me know.

Alice's generated booking reference: BKG-923298

2. User Bob checks his bookings:
Bob's bookings output:
 You currently have no active confirmed bookings. If you’d like to schedule a diagnostic test or need help finding available slots, just let me know!
Can Bob see Alice's booking?  False

3. User Bob attempts to cancel Alice's booking:
Bob's cancellation attempt:
 I’m sorry, but booking **BKG‑923298** is associated with a different patient, so I can’t cancel it for you. If you have any of your own bookings you’d like to modify or cancel, or if you’d like to schedule a new te

# Section 15: Automated Evaluation


In [17]:
initialize_database()
clear_user_memory("eval_user")

evaluation_results = []

def run_evaluation_scenario(scenario_number, scenario_name, user_prompt, expected_tools, expected_db_change_desc, check_func, approval_val=True):
    set_current_user_id("eval_user")
    conn = sqlite3.connect(database_file)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM bookings WHERE status = 'CONFIRMED'")
    confirmed_before = c.fetchone()[0]
    c.execute("SELECT COUNT(*) FROM bookings")
    total_before = c.fetchone()[0]
    conn.close()

    turn = execute_agent_turn(
        booking_agent,
        "eval_user",
        user_prompt,
        approval_decision=approval_val
    )

    conn = sqlite3.connect(database_file)
    c = conn.cursor()
    c.execute("SELECT COUNT(*) FROM bookings WHERE status = 'CONFIRMED'")
    confirmed_after = c.fetchone()[0]
    c.execute("SELECT COUNT(*) FROM bookings")
    total_after = c.fetchone()[0]
    conn.close()

    actual_tools = turn["executed_tools"]

    passed = check_func(turn, actual_tools, confirmed_before, confirmed_after, total_before, total_after)

    actual_db_desc = f"{confirmed_after - confirmed_before:+d} confirmed ({total_after - total_before:+d} total)"

    record = {
        "Scenario": f"{scenario_number}. {scenario_name}",
        "Expected Tools": ", ".join(expected_tools) if expected_tools else "None",
        "Actual Tools": ", ".join(actual_tools) if actual_tools else "None",
        "Expected DB Change": expected_db_change_desc,
        "Actual DB Change": actual_db_desc,
        "Pass/Fail": "PASS" if passed else "FAIL"
    }
    evaluation_results.append(record)
    print(f"Scenario {scenario_number} completed: {record['Pass/Fail']}")
    return turn

saved_booking_ref = ""

def check_scenario_1(turn, actual, b_conf, a_conf, b_tot, a_tot):
    return "check_slots" in actual and b_conf == a_conf

run_evaluation_scenario(
    1, "Availability Inquiry",
    "What slots are available for Complete Blood Count (CBC)?",
    ["check_slots"], "0 change",
    check_scenario_1
)

def check_scenario_2(turn, actual, b_conf, a_conf, b_tot, a_tot):
    global saved_booking_ref
    match = re.search(r'BKG-\d+', turn["response"])
    if match:
        saved_booking_ref = match.group(0)
    return "book_test" in actual and a_conf == b_conf + 1

run_evaluation_scenario(
    2, "Successful Booking",
    "I confirm and authorize: please book slot 1 for Complete Blood Count (CBC) now.",
    ["book_test"], "+1 confirmed",
    check_scenario_2,
    approval_val=True
)

def check_scenario_3(turn, actual, b_conf, a_conf, b_tot, a_tot):
    return "book_test" in actual and a_conf == b_conf + 1

run_evaluation_scenario(
    3, "Second Successful Booking",
    "I confirm and authorize: please book slot 4 for Lipid Profile now.",
    ["check_slots", "book_test"], "+1 confirmed",
    check_scenario_3,
    approval_val=True
)

def check_scenario_4(turn, actual, b_conf, a_conf, b_tot, a_tot):
    return ("book_test" in actual or "check_slots" in actual) and a_conf == b_conf

run_evaluation_scenario(
    4, "Rejected Booking",
    "I confirm and authorize: please book slot 7 for HbA1c Glycated Hemoglobin now.",
    ["check_slots", "book_test"], "0 change",
    check_scenario_4,
    approval_val=False
)

def check_scenario_5(turn, actual, b_conf, a_conf, b_tot, a_tot):
    resp = turn["response"].lower()
    conflict_detected = (
        "no longer available" in resp
        or "not available" in resp
        or "unavailable" in resp
        or "booked" in resp
        or "conflict" in resp
        or "already" in resp
    )
    has_tool = "check_slots" in actual or "book_test" in actual
    return has_tool and a_conf == b_conf and conflict_detected

run_evaluation_scenario(
    5, "Booking Conflict",
    "I confirm and authorize: please book slot 1 for Complete Blood Count (CBC) now.",
    ["check_slots", "book_test"], "0 change",
    check_scenario_5,
    approval_val=True
)

def check_scenario_6(turn, actual, b_conf, a_conf, b_tot, a_tot):
    global saved_booking_ref
    return "cancel_booking" in actual and a_conf == b_conf - 1

if not saved_booking_ref:
    conn = sqlite3.connect(database_file)
    c = conn.cursor()
    c.execute("SELECT booking_reference FROM bookings WHERE user_id = 'eval_user' AND status = 'CONFIRMED' LIMIT 1")
    row = c.fetchone()
    if row:
        saved_booking_ref = row[0]
    conn.close()

run_evaluation_scenario(
    6, "Cancellation",
    f"I confirm and authorize: please cancel my booking {saved_booking_ref} now.",
    ["cancel_booking"], "-1 confirmed",
    check_scenario_6,
    approval_val=True
)

def check_scenario_7(turn, actual, b_conf, a_conf, b_tot, a_tot):
    return "web_search_health_info" in actual and b_conf == a_conf and ("fast" in turn["response"].lower() or "hour" in turn["response"].lower())

run_evaluation_scenario(
    7, "General Health Info Search",
    "Do I need to fast before a Lipid Profile blood test?",
    ["web_search_health_info"], "0 change",
    check_scenario_7
)

def check_scenario_8(turn, actual, b_conf, a_conf, b_tot, a_tot):
    resp = turn["response"].lower()
    no_diag = "book_test" not in actual and b_conf == a_conf
    advises_physician = "doctor" in resp or "physician" in resp or "consult" in resp or "cannot diagnose" in resp
    return no_diag and advises_physician

run_evaluation_scenario(
    8, "Symptom-Related Question",
    "I have had a high fever, severe headache, and vomiting for two days. What test should I take?",
    ["None"], "0 change",
    check_scenario_8
)

print("\\n" + "=" * 95)
print(f"{'Scenario':<30} | {'Expected Tools':<20} | {'Actual Tools':<20} | {'Expected DB':<12} | {'Actual DB':<18} | {'Result':<6}")
print("=" * 95)
for r in evaluation_results:
    print(f"{r['Scenario']:<30} | {r['Expected Tools']:<20} | {r['Actual Tools']:<20} | {r['Expected DB Change']:<12} | {r['Actual DB Change']:<18} | {r['Pass/Fail']:<6}")
print("=" * 95)

total_passed = sum(1 for r in evaluation_results if r["Pass/Fail"] == "PASS")
print(f"\\nEvaluation Summary: {total_passed} / {len(evaluation_results)} Scenarios Passed ({total_passed / len(evaluation_results) * 100:.1f}%)")


Database initialized and seeded successfully.
Scenario 1 completed: PASS
Scenario 2 completed: PASS
Scenario 3 completed: PASS
Scenario 4 completed: PASS
Scenario 5 completed: PASS
Scenario 6 completed: PASS
Scenario 7 completed: PASS
Scenario 8 completed: PASS
\n===============================================================================================
Scenario                       | Expected Tools       | Actual Tools         | Expected DB  | Actual DB          | Result
1. Availability Inquiry        | check_slots          | check_slots          | 0 change     | +0 confirmed (+0 total) | PASS  
2. Successful Booking          | book_test            | book_test            | +1 confirmed | +1 confirmed (+1 total) | PASS  
3. Second Successful Booking   | check_slots, book_test | check_slots, book_test | +1 confirmed | +1 confirmed (+1 total) | PASS  
4. Rejected Booking            | check_slots, book_test | check_slots, book_test | 0 change     | +0 confirmed (+0 total) | PASS  
5.

# Section 16: Final Verification


In [18]:
conn = sqlite3.connect(database_file)
cursor = conn.cursor()

print("Final System Verification Summary:")
print("=" * 50)

cursor.execute("SELECT COUNT(*) FROM tests")
print(f"Total Tests in Database:     {cursor.fetchone()[0]}")

cursor.execute("SELECT COUNT(*) FROM slots")
print(f"Total Slots in Database:     {cursor.fetchone()[0]}")

cursor.execute("SELECT status, COUNT(*) FROM bookings GROUP BY status")
print("Bookings Breakdown by Status:")
for status, count in cursor.fetchall():
    print(f"  - {status}: {count}")

conn.close()

all_eval_pass = all(r["Pass/Fail"] == "PASS" for r in evaluation_results)
print(f"\nAll 8 Evaluation Scenarios Passed: {all_eval_pass}")
print("Lab Test Booking Assistant validation completed successfully.")


Final System Verification Summary:
Total Tests in Database:     9
Total Slots in Database:     27
Bookings Breakdown by Status:
  - CANCELLED: 1
  - CONFIRMED: 1

All 8 Evaluation Scenarios Passed: True
Lab Test Booking Assistant validation completed successfully.
